# CS-320 · YOLOv1 project · Hassaan — Part C: Augmentation ablation (RQ2)
**Question:** *How does data augmentation affect performance when training on Penn-Fudan?*  
**Design.** Fine-tune the same Ultralytics YOLO model on the same data with identical settings and seeds; the **only** difference is the augmentation hyper-parameters:

* `no_aug` — HSV, flips, scale, translate, shear, rotation, perspective, mosaic, mixup, copy-paste, erasing all set to 0
* `default_aug` — Ultralytics defaults

Each variant is run with 3 seeds (26 test images → single runs are very noisy). The checkpoint is chosen on **val**; the **test** split is evaluated once per run. Never tune on test.

## 0 · Setup

In [ ]:
import os, sys, json, shutil, random
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:                                   # Drive is used to keep results if Colab disconnects
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/yolo_project'
    os.makedirs(DRIVE, exist_ok=True)
except Exception:
    DRIVE = None
print('Drive folder:', DRIVE)

In [ ]:
!pip install -q ultralytics
import torch, ultralytics
print('ultralytics', ultralytics.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable Runtime > Change runtime type > GPU')

### Project modules
The cells below write the `yolotools` package to disk (same files that live in the GitHub repo under `yolotools/`). Run them once; nothing needs to be edited.

In [ ]:
!mkdir -p /content/cs320-yolov1/yolotools

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/__init__.py
"""CS-320 YOLOv1 project - Hassaan (Data lead) toolkit.

Modules
-------
boxes         box format conversions and IoU
penn_fudan    Penn-Fudan download, PASCAL->YOLO conversion, splits, dataset writer
quality       the 10 dataset quality checks + report
viz           dataset figures (sample grid, mask instances, label check, stats)
augment       box-aware augmentation (HSV, flip, scale+translate)
yolo_dataset  loader for the YOLO-format Penn-Fudan folder
voc2007       PASCAL VOC 2007 download + loader + YOLOv1 7x7xK target encoder
nms           NMS, class-aware NMS, YOLOv1 output decoding
metrics       VOC-style AP / mAP@IoU (07 11-point and all-point), P/R at threshold
evaluate      glue: run a YOLOv1-style model over a dataset and compute mAP
ablation      Ultralytics augmentation on/off ablation helpers
"""
__version__ = "1.0.0"

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/boxes.py
"""Box utilities (NumPy only).

Conventions
-----------
xyxy : (x1, y1, x2, y2)   continuous edge coordinates (x2 > x1)
xywh : (xc, yc, w, h)     centre + size
IoU is computed on continuous coordinates. Because IoU is invariant to scaling each
axis separately, it gives identical values on pixel or on normalised (0-1) boxes.
"""
import numpy as np


def xywh_to_xyxy(b):
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    return np.stack([b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2,
                     b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], axis=1)


def xyxy_to_xywh(b):
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    return np.stack([(b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2,
                     b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]], axis=1)


def box_area(b):
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    return np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)


def iou_matrix(a, b):
    """Pairwise IoU between a (N,4) and b (M,4) xyxy boxes -> (N,M)."""
    a = np.asarray(a, dtype=np.float64).reshape(-1, 4)
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0])
    iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2])
    iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    union = box_area(a)[:, None] + box_area(b)[None, :] - inter
    return np.where(union > 0, inter / np.maximum(union, 1e-12), 0.0)


def iou_one_to_many(box, boxes):
    return iou_matrix(np.asarray(box).reshape(1, 4), boxes)[0]

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/penn_fudan.py
"""Penn-Fudan Pedestrian -> YOLO-format dataset (download, parse, convert, split, write).

Official PASCAL annotation boxes are 1-indexed INCLUSIVE pixel coordinates. They are converted to
continuous edges  (xmin-1, ymin-1, xmax, ymax)  and then to normalised YOLO `class xc yc w h`.
Every box is cross-checked against the box derived from the instance mask.
"""
import csv, json, os, re, shutil, urllib.request, zipfile
import numpy as np
from PIL import Image
from .boxes import iou_matrix, xyxy_to_xywh

URL = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"
SEED = 42
RATIOS = (0.70, 0.15, 0.15)
EXPECTED = {"images": 170, "instances": 423}          # published Penn-Fudan totals
MIN_MASK_IOU_WARN = 0.70

_BOX_RE = re.compile(r"Bounding box for object\s+\d+.*?:\s*\((\d+),\s*(\d+)\)\s*-\s*\((\d+),\s*(\d+)\)")
_SIZE_RE = re.compile(r"Image size \(X x Y x C\)\s*:\s*(\d+)\s*x\s*(\d+)\s*x\s*(\d+)")


def download_penn_fudan(dest="/content", url=URL):
    """Download + unzip; returns the dataset root (folder containing PNGImages/)."""
    os.makedirs(dest, exist_ok=True)
    root = os.path.join(dest, "PennFudanPed")
    if os.path.isdir(os.path.join(root, "PNGImages")):
        return root
    zpath = os.path.join(dest, "PennFudanPed.zip")
    urllib.request.urlretrieve(url, zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(dest)
    return root


def parse_annotation(path):
    """-> dict(width, height, boxes=[(xmin,ymin,xmax,ymax)] in 1-indexed inclusive pixels)."""
    txt = open(path, encoding="latin-1").read()
    m = _SIZE_RE.search(txt)
    if not m:
        raise ValueError(f"no image size in {path}")
    boxes = [tuple(int(v) for v in g) for g in _BOX_RE.findall(txt)]
    return {"width": int(m.group(1)), "height": int(m.group(2)), "boxes": boxes}


def annotation_to_edges(b):
    xmin, ymin, xmax, ymax = b
    return (xmin - 1, ymin - 1, xmax, ymax)


def mask_instance_edges(mask_path):
    """Instance boxes from the mask (0 = background, 1..N = pedestrians) as continuous edges."""
    mask = np.array(Image.open(mask_path))
    out = []
    for i in np.unique(mask):
        if i == 0:
            continue
        ys, xs = np.where(mask == i)
        out.append((xs.min(), ys.min(), xs.max() + 1, ys.max() + 1))
    return np.array(out, dtype=np.float64).reshape(-1, 4)


def edges_to_yolo(edges, W, H):
    xc, yc, w, h = xyxy_to_xywh(np.asarray(edges, float).reshape(-1, 4)).T
    return np.stack([xc / W, yc / H, w / W, h / H], axis=1)


def yolo_to_edges(yolo, W, H):
    yolo = np.asarray(yolo, float).reshape(-1, 4)
    return np.stack([(yolo[:, 0] - yolo[:, 2] / 2) * W, (yolo[:, 1] - yolo[:, 3] / 2) * H,
                     (yolo[:, 0] + yolo[:, 2] / 2) * W, (yolo[:, 1] + yolo[:, 3] / 2) * H], axis=1)


def source_set(name):
    return "Penn" if name.startswith("PennPed") else "Fudan"


def build_records(root):
    """Parse + validate everything. Returns (records, issues, warnings)."""
    d_img, d_ann, d_msk = (os.path.join(root, d) for d in ("PNGImages", "Annotation", "PedMasks"))
    imgs = sorted(f for f in os.listdir(d_img) if f.lower().endswith(".png"))
    anns = {os.path.splitext(f)[0] for f in os.listdir(d_ann) if f.endswith(".txt")}
    msks = {f[:-len("_mask.png")] for f in os.listdir(d_msk) if f.endswith("_mask.png")}
    stems = {os.path.splitext(f)[0] for f in imgs}
    issues = {"unreadable_images": [], "missing_masks": [], "missing_annotations": [], "orphan_masks": [],
              "orphan_annotations": [], "size_mismatch": [], "invalid_boxes": [], "conversion_failures": []}
    warnings = []
    issues["missing_annotations"] = sorted(stems - anns)
    issues["missing_masks"] = sorted(stems - msks)
    issues["orphan_annotations"] = sorted(anns - stems)
    issues["orphan_masks"] = sorted(msks - stems)
    records = []
    for f in imgs:
        stem = os.path.splitext(f)[0]
        if stem in issues["missing_annotations"]:
            continue
        try:
            with Image.open(os.path.join(d_img, f)) as im:
                im.verify()
            with Image.open(os.path.join(d_img, f)) as im:
                W, H = im.size
        except Exception as e:                                  # noqa: BLE001
            issues["unreadable_images"].append(f"{f}: {e}")
            continue
        try:
            ann = parse_annotation(os.path.join(d_ann, stem + ".txt"))
        except Exception as e:                                  # noqa: BLE001
            issues["conversion_failures"].append(f"{f}: {e}")
            continue
        if (ann["width"], ann["height"]) != (W, H):
            issues["size_mismatch"].append(f"{f}: annotation {ann['width']}x{ann['height']} vs image {W}x{H}")
        edges = np.array([annotation_to_edges(b) for b in ann["boxes"]], float).reshape(-1, 4)
        bad = [i for i, e in enumerate(edges) if e[2] <= e[0] or e[3] <= e[1]]
        if bad:
            issues["invalid_boxes"].append(f"{f}: degenerate box index {bad}")
            edges = np.delete(edges, bad, axis=0)
        clipped = np.clip(edges, 0, [W, H, W, H])
        if len(edges) and not np.allclose(clipped, edges):
            warnings.append(f"{f}: box extended outside image and was clipped")
        edges = clipped
        miou = None
        if stem not in issues["missing_masks"]:
            mb = mask_instance_edges(os.path.join(d_msk, stem + "_mask.png"))
            if len(mb) != len(edges):
                warnings.append(f"{f}: {len(edges)} annotation boxes vs {len(mb)} mask instances")
            if len(mb) and len(edges):
                best = iou_matrix(edges, mb).max(axis=1)
                miou = float(best.min())
                if miou < MIN_MASK_IOU_WARN:
                    warnings.append(f"{f}: lowest annotation-vs-mask IoU {miou:.2f}")
        records.append({"filename": f, "source_set": source_set(f), "width": W, "height": H,
                        "edges": edges, "min_mask_iou": miou})
    return records, issues, warnings


def make_split(filenames, seed=SEED, ratios=RATIOS):
    """Image-level random split. Floor for train/val, remainder to test (170 -> 119/25/26)."""
    names = sorted(filenames)
    perm = np.random.RandomState(seed).permutation(len(names))
    n_tr = int(np.floor(ratios[0] * len(names) + 1e-9))   # epsilon: 0.7*170 == 118.99999...
    n_va = int(np.floor(ratios[1] * len(names) + 1e-9))
    split = {}
    for rank, i in enumerate(perm):
        split[names[i]] = "train" if rank < n_tr else ("val" if rank < n_tr + n_va else "test")
    return split


def load_manifest_split(path):
    with open(path, newline="") as f:
        return {r["filename"]: r["split"] for r in csv.DictReader(f)}


DATASET_README = """# PennFudanYOLO (Member 1 handover)
- Source: official Penn-Fudan zip ({url})
- Labels: YOLO detection format `0 xc yc w h` (normalised by each image's own size), class 0 = pedestrian
- Boxes: official PASCAL annotation boxes (1-indexed inclusive -> converted), cross-checked against mask instance boxes
- Split: image level 70/15/15, seed {seed}, see split_manifest.csv
- Images kept at original resolution (resizing is done by the training code; normalised labels are unaffected)
- Do NOT tune hyper-parameters on the test split
- Update `path:` in data.yaml to wherever this folder lives on your machine/Colab
"""


def write_dataset(root, out_dir, records, split, seed=SEED, issues=None, warnings=None):
    """Write images/, labels/, split_manifest.csv, dataset_stats.csv, data.yaml, README_dataset.md, conversion_log.txt."""
    if os.path.isdir(out_dir):
        for sub in ("images", "labels"):
            shutil.rmtree(os.path.join(out_dir, sub), ignore_errors=True)
    for s in ("train", "val", "test"):
        os.makedirs(os.path.join(out_dir, "images", s), exist_ok=True)
        os.makedirs(os.path.join(out_dir, "labels", s), exist_ok=True)
    rows = []
    for r in records:
        s = split[r["filename"]]
        shutil.copy2(os.path.join(root, "PNGImages", r["filename"]), os.path.join(out_dir, "images", s, r["filename"]))
        yolo = edges_to_yolo(r["edges"], r["width"], r["height"])
        with open(os.path.join(out_dir, "labels", s, os.path.splitext(r["filename"])[0] + ".txt"), "w") as f:
            for xc, yc, w, h in yolo:
                f.write(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
        rows.append({"filename": r["filename"], "split": s, "source_set": r["source_set"], "width": r["width"],
                     "height": r["height"], "n_pedestrians": len(yolo), "label_source": "annotation"})
    rows.sort(key=lambda x: x["filename"])
    with open(os.path.join(out_dir, "split_manifest.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    stats = [("Total", len(rows), sum(r["n_pedestrians"] for r in rows))]
    for s in ("train", "val", "test"):
        sel = [r for r in rows if r["split"] == s]
        stats.append((s.capitalize(), len(sel), sum(r["n_pedestrians"] for r in sel)))
    with open(os.path.join(out_dir, "dataset_stats.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["Metric", "Images", "Pedestrian instances"]); w.writerows(stats)
    with open(os.path.join(out_dir, "data.yaml"), "w") as f:
        f.write("# Auto-generated by Member 1 notebook (Penn-Fudan, single class)\n"
                f"path: {os.path.abspath(out_dir)}\ntrain: images/train\nval: images/val\ntest: images/test\n"
                "nc: 1\nnames:\n  0: pedestrian\n")
    with open(os.path.join(out_dir, "README_dataset.md"), "w") as f:
        f.write(DATASET_README.format(url=URL, seed=seed))
    with open(os.path.join(out_dir, "conversion_log.txt"), "w") as f:
        f.write("ISSUES\n" + json.dumps(issues or {}, indent=2) + "\n\nWARNINGS\n" + "\n".join(warnings or []) + "\n")
    return rows, stats


def prepare(root, out_dir, manifest_path=None, seed=SEED):
    """Full pipeline. If manifest_path is given its split is reused (keeps team results comparable)."""
    records, issues, warnings = build_records(root)
    names = [r["filename"] for r in records]
    if manifest_path and os.path.exists(manifest_path):
        split = load_manifest_split(manifest_path)
        missing = set(names) - set(split)
        if missing:
            raise ValueError(f"manifest lacks {len(missing)} images, e.g. {sorted(missing)[:3]}")
        split = {n: split[n] for n in names}
    else:
        split = make_split(names, seed)
    rows, stats = write_dataset(root, out_dir, records, split, seed, issues, warnings)
    return records, split, issues, warnings, stats

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/ablation.py
"""Augmentation ON vs OFF ablation with Ultralytics YOLO on Penn-Fudan (RQ2).

Design
------
* Same model weights, data, epochs, image size, batch, optimiser settings and seeds for every variant;
  ONLY the augmentation hyper-parameters differ.
* `default_aug`  = Ultralytics defaults (HSV, flip, scale, translate, mosaic, ...).
* `no_aug`       = every stochastic augmentation set to zero (images are only letter-boxed/resized, which is
                   preprocessing, not augmentation).
* Several seeds per variant (tiny dataset -> high variance). Test split evaluated ONCE per run with the
  checkpoint chosen on the VALIDATION split (Ultralytics best.pt).
"""
import csv, json, os
import numpy as np

# every stochastic-augmentation argument Ultralytics knows about, set to "off"
AUG_OFF = dict(hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, degrees=0.0, translate=0.0, scale=0.0, shear=0.0,
               perspective=0.0, flipud=0.0, fliplr=0.0, bgr=0.0, mosaic=0.0, mixup=0.0, cutmix=0.0,
               copy_paste=0.0, erasing=0.0, close_mosaic=0)
# optional single-component variants ("one at a time")
EXTRA_VARIANTS = {
    "flip_only": dict(AUG_OFF, fliplr=0.5),
    "hsv_only": dict(AUG_OFF, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4),
    "geometric_only": dict(AUG_OFF, translate=0.1, scale=0.5),
    "mosaic_only": dict(AUG_OFF, mosaic=1.0),
}
VARIANTS = {"no_aug": AUG_OFF, "default_aug": {}}


def supported(overrides):
    """Drop arguments the installed Ultralytics version does not know (e.g. cutmix on older releases)."""
    try:
        from ultralytics.cfg import DEFAULT_CFG_DICT
        return {k: v for k, v in overrides.items() if k in DEFAULT_CFG_DICT}
    except Exception:                                      # noqa: BLE001
        return dict(overrides)


def run_one(variant, overrides, seed, data_yaml, model_name="yolo11n.pt", epochs=50, imgsz=640, batch=16,
            device=0, project="runs/aug_ablation", workers=2):
    from ultralytics import YOLO
    name = f"{variant}_seed{seed}"
    model = YOLO(model_name)
    model.train(data=data_yaml, epochs=epochs, imgsz=imgsz, batch=batch, seed=seed, deterministic=True,
                device=device, project=project, name=name, exist_ok=True, plots=False, verbose=False,
                workers=workers, patience=epochs, **supported(overrides))
    best = YOLO(os.path.join(project, name, "weights", "best.pt"))
    out = {"variant": variant, "seed": seed, "model": model_name, "epochs": epochs, "imgsz": imgsz}
    for split in ("val", "test"):
        m = best.val(data=data_yaml, split=split, imgsz=imgsz, batch=batch, device=device, conf=0.001,
                     iou=0.6, plots=False, verbose=False)
        out.update({f"{split}_precision": float(m.box.mp), f"{split}_recall": float(m.box.mr),
                    f"{split}_map50": float(m.box.map50), f"{split}_map50_95": float(m.box.map)})
        if split == "test":
            out.update({f"speed_{k}_ms": float(v) for k, v in m.speed.items()})
    return out


def run_ablation(data_yaml, variants=None, seeds=(0, 1, 2), out_csv="aug_ablation_runs.csv", **kw):
    """Run every variant x seed, appending to out_csv after each run (resumable if Colab disconnects)."""
    variants = variants or VARIANTS
    done = set()
    rows = []
    if os.path.exists(out_csv):
        rows = list(csv.DictReader(open(out_csv)))
        done = {(r["variant"], int(r["seed"])) for r in rows}
    for v, ov in variants.items():
        for s in seeds:
            if (v, s) in done:
                print(f"skip {v} seed {s} (already in {out_csv})"); continue
            print(f"=== {v} | seed {s} ===")
            r = run_one(v, ov, s, data_yaml, **kw)
            rows.append(r)
            with open(out_csv, "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=sorted({k for x in rows for k in x}))
                w.writeheader(); w.writerows(rows)
    return rows


def summarise(rows, metrics=("test_map50", "test_map50_95", "test_precision", "test_recall",
                             "val_map50", "val_map50_95")):
    """-> {variant: {metric: (mean, std, n)}} (sample std, ddof=1 when n>1)."""
    out = {}
    for v in sorted({r["variant"] for r in rows}):
        sel = [r for r in rows if r["variant"] == v]
        out[v] = {m: (float(np.mean([float(r[m]) for r in sel])),
                      float(np.std([float(r[m]) for r in sel], ddof=1)) if len(sel) > 1 else 0.0, len(sel)) for m in metrics}
    return out


def markdown_table(summary, metrics=("test_map50", "test_map50_95", "test_precision", "test_recall")):
    head = "| Variant | n | " + " | ".join(metrics) + " |\n|---|---|" + "---|" * len(metrics)
    lines = [head]
    for v, d in summary.items():
        lines.append(f"| {v} | {d[metrics[0]][2]} | " + " | ".join(f"{d[m][0]:.3f} ± {d[m][1]:.3f}" for m in metrics) + " |")
    return "\n".join(lines)

In [ ]:
import sys
sys.path.insert(0, '/content/cs320-yolov1')
import yolotools
print('yolotools', yolotools.__version__, 'ready')

## 1 · Get the dataset (from Part A)
Order tried: folder already in `/content` → `PennFudanYOLO_full.zip` on Drive → rebuild from the original download using the team manifest.

In [ ]:
import glob, subprocess
OUT = '/content/PennFudanYOLO'
def n_images(p): return sum(len(glob.glob(f'{p}/images/{s}/*.png')) for s in ('train', 'val', 'test'))
if not (os.path.isdir(OUT) and n_images(OUT) == 170):
    zp = f'{DRIVE}/PennFudanYOLO_full.zip' if DRIVE else None
    if zp and os.path.exists(zp):
        shutil.unpack_archive(zp, '/content'); print('unzipped from Drive')
    else:
        from yolotools import penn_fudan as pf
        subprocess.run('rm -rf /content/team_repo && git clone -q --depth 1 https://github.com/ayeshaalamexe/cs320-yolov1.git /content/team_repo', shell=True)
        man = '/content/team_repo/PennFudanYOLO/split_manifest.csv'
        pf.prepare(pf.download_penn_fudan('/content'), OUT, man if os.path.exists(man) else None); print('rebuilt dataset')
assert n_images(OUT) == 170, n_images(OUT)
# make data.yaml point to this machine's folder
yml = open(f'{OUT}/data.yaml').read().splitlines()
yml = [f'path: {OUT}' if l.startswith('path:') else l for l in yml]
open(f'{OUT}/data.yaml', 'w').write('\n'.join(yml) + '\n'); DATA = f'{OUT}/data.yaml'
print(open(DATA).read())

## 2 · Configuration
Keep these identical for every variant. 50 epochs ≈ a few minutes per run on a Colab T4.

In [ ]:
from yolotools import ablation
MODEL    = 'yolo11n.pt'         # COCO-pretrained, fine-tuned (matches the proposal). Use the .yaml name for a from-scratch run.
EPOCHS, IMGSZ, BATCH = 50, 640, 16
SEEDS    = (0, 1, 2)
RUN_EXTRA = False               # True also runs flip_only / hsv_only / geometric_only / mosaic_only ("one at a time")
VARIANTS = dict(ablation.VARIANTS)
if RUN_EXTRA: VARIANTS.update(ablation.EXTRA_VARIANTS)
RES_DIR  = f'{DRIVE}/ablation' if DRIVE else '/content/ablation'; os.makedirs(RES_DIR, exist_ok=True)
CSV      = f'{RES_DIR}/aug_ablation_runs_{MODEL.split(".")[0]}.csv'
print({k: ablation.supported(v) for k, v in VARIANTS.items()})

### Smoke test (1 epoch, 1 run) - catches setup problems in ~1 minute

In [ ]:
_ = ablation.run_one('smoke', ablation.AUG_OFF, 0, DATA, MODEL, epochs=1, imgsz=IMGSZ, batch=BATCH, project='/content/runs/smoke')
print({k: round(v, 3) if isinstance(v, float) else v for k, v in _.items()})

## 3 · Run the ablation
Results are appended to a CSV on Drive after every run, so if Colab disconnects just re-run this cell — finished runs are skipped.

In [ ]:
rows = ablation.run_ablation(DATA, VARIANTS, SEEDS, out_csv=CSV, model_name=MODEL, epochs=EPOCHS, imgsz=IMGSZ,
                             batch=BATCH, project='/content/runs/aug_ablation')
import pandas as pd
df = pd.read_csv(CSV); df

## 4 · Results

In [ ]:
summary = ablation.summarise(df.to_dict('records'))
table = ablation.markdown_table(summary); print(table)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, m in zip(ax, ['test_map50', 'test_map50_95']):
    names = list(summary); a.bar(names, [summary[v][m][0] for v in names], yerr=[summary[v][m][1] for v in names], capsize=6, color=['#999', '#4363d8', '#3cb44b', '#f58231', '#911eb4', '#008080'][:len(names)])
    for k, v in enumerate(names):
        pts = df[df.variant == v][m]; a.scatter([k] * len(pts), pts, color='k', zorder=3, s=18)
    a.set_title(m + ' (bar = mean ± sd, dots = seeds)'); a.set_ylim(max(0, df[m].min() - .05), min(1, df[m].max() + .03)); a.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.savefig(f'{RES_DIR}/aug_ablation_bars.png', dpi=120); plt.show()

In [ ]:
# learning curves (val mAP50 per epoch, seed 0) - shows over-fitting / convergence differences
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for v in VARIANTS:
    p = f'/content/runs/aug_ablation/{v}_seed0/results.csv'
    if not os.path.exists(p): continue
    r = pd.read_csv(p); r.columns = [c.strip() for c in r.columns]
    ax[0].plot(r['epoch'], r['metrics/mAP50(B)'], label=v); ax[1].plot(r['epoch'], r['val/box_loss'], label=v)
ax[0].set_title('val mAP50 vs epoch'); ax[1].set_title('val box loss vs epoch'); ax[0].legend(); ax[0].set_xlabel('epoch'); ax[1].set_xlabel('epoch')
plt.tight_layout(); plt.savefig(f'{RES_DIR}/aug_ablation_curves.png', dpi=120); plt.show()

In [ ]:
# qualitative: same 4 test images, predictions of the best checkpoint of each variant (seed 0)
from ultralytics import YOLO
from PIL import Image
test_imgs = sorted(glob.glob(f'{OUT}/images/test/*.png'))[:4]
fig, axes = plt.subplots(len(VARIANTS), 4, figsize=(18, 4.2 * len(VARIANTS)), squeeze=False)
for r_, v in enumerate(VARIANTS):
    w = f'/content/runs/aug_ablation/{v}_seed0/weights/best.pt'
    if not os.path.exists(w): continue
    m = YOLO(w)
    for c_, p in enumerate(test_imgs):
        axes[r_, c_].imshow(m.predict(p, imgsz=IMGSZ, conf=0.25, verbose=False)[0].plot()[..., ::-1]); axes[r_, c_].axis('off'); axes[r_, c_].set_title(f'{v} | {os.path.basename(p)}', fontsize=9)
plt.tight_layout(); plt.savefig(f'{RES_DIR}/aug_ablation_predictions.png', dpi=110); plt.show()

## 5 · Auto-generated report text
Numbers are filled in from the CSV. The comparison sentence is deliberately cautious: with 26 test images / 73 pedestrians and 3 seeds, a difference smaller than the seed-to-seed spread is **not** evidence of an effect.

In [ ]:
m = 'test_map50'; a, b = summary['default_aug'][m], summary['no_aug'][m]
diff, noise = a[0] - b[0], max(a[1], b[1])
verdict = ('larger than' if abs(diff) > 2 * noise else 'comparable to' if abs(diff) > noise else 'within') + ' the seed-to-seed standard deviation'
text = f"""# Augmentation ablation (RQ2) - Penn-Fudan, {MODEL}, {EPOCHS} epochs, imgsz {IMGSZ}, batch {BATCH}, seeds {list(SEEDS)}

All variants: same pretrained weights, data split (119/25/26 images), epochs, image size, batch size and seeds; only augmentation
arguments differ. Best checkpoint chosen on val; test evaluated once per run (conf=0.001, NMS IoU=0.6).

{table}

mean ± sample sd over {a[2]} seeds. On test mAP@0.5, default augmentation scores {a[0]:.3f} versus {b[0]:.3f} without
augmentation (difference {diff:+.3f}), which is {verdict} ({noise:.3f}).
Per-run numbers: aug_ablation_runs CSV. Figures: aug_ablation_bars.png, aug_ablation_curves.png, aug_ablation_predictions.png.

Interpretation notes (check them against the figures before writing the report):
* A COCO-pretrained detector already knows 'person', so Penn-Fudan can sit close to a performance ceiling and hide augmentation effects; the from-scratch run (MODEL = 'yolo11n.yaml') is the stricter test of RQ2.
* Compare val-loss curves: if no_aug's validation loss rises while training loss falls, that is over-fitting that augmentation is regularising.
* The test split is tiny (26 images); report seeds, not a single run.
"""
open(f'{RES_DIR}/aug_ablation_report.md', 'w').write(text); print(text)

**Optional, stricter run.** Set `MODEL = 'yolo11n.yaml'` (from scratch), `EPOCHS = 150`, re-run sections 2-5 — this writes a separate CSV and shows augmentation's effect without the COCO prior.

Copy `ablation/*.md/png/csv` into the repo's `results/` folder.